# 🌍 Fine-Tuning NMT con ByT5 (Wolof / Español / Italiano / Suajili)

**Objetivo:** pipeline completo de traducción automática neuronal (NMT) para
lenguas de bajo recurso: **carga → limpieza → tokenización → fine-tuning de
ByT5 → evaluación (BLEU/chrF)**.

Este notebook es la versión limpia y reproducible del pipeline legacy:
- **100% PyTorch** (sin TensorFlow/Keras)
- Sin rutas de Colab (`/content/`) ni Windows (`C:\`), sin `drive.mount()`
- Datos configurables con `DATA_DIR` (CSV, JSONL o pares `.src`/`.tgt`)
- Carga desde **PostgreSQL** opcional (credenciales vía variables de entorno)

**Modelo:** `google/byt5-small` — tokenización a nivel de **bytes**, ideal para
lenguas sin tokenizador propio (wolof, fula, bambara...).

**Requisitos:** Python 3.10+, PyTorch, `transformers`, `datasets`, `sacrebleu`, `pandas`.

---
## 0. Instalación

```bash
pip install torch transformers datasets sacrebleu pandas tqdm
# Solo si quieres cargar datos desde PostgreSQL:
pip install sqlalchemy psycopg2-binary
```

> 🔑 **Modelos gated:** si usas un modelo que requiere autenticación, define
> `export HF_TOKEN=hf_...` en el entorno. Nunca se hardcodean tokens.

In [ ]:
# === IMPORTS Y DISPOSITIVO ===
import os
import json
import random
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import (
    ByT5Tokenizer,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"HF_TOKEN definido: {bool(os.environ.get('HF_TOKEN'))}")

---
## 1. Configuración

`DATA_DIR` apunta a la carpeta con los datos paralelos. El resto de
hiperparámetros viven en el dataclass `Config`.

In [ ]:
# === CONFIGURACIÓN ===
DATA_DIR = Path("./data/nmt")          # <- CAMBIA ESTO
OUTPUT_DIR = Path("./models/nmt_byt5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    # --- Modelo ---
    model_name: str = "google/byt5-small"
    max_length: int = 128

    # --- Pares de lenguas (ISO 639-1) ---
    # Cada tupla define una dirección: (origen, destino)
    language_pairs: List[Tuple[str, str]] = None
    bidirectional: bool = True          # añade también la dirección inversa

    # --- Datos ---
    max_train_samples: Optional[int] = None   # límite para pruebas rápidas
    val_ratio: float = 0.05
    test_ratio: float = 0.05
    min_len: int = 3                    # filtro de longitud de frase
    max_len: int = 75

    # --- Entrenamiento ---
    num_epochs: int = 8
    batch_size: int = 16
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    gradient_accumulation_steps: int = 2
    warmup_steps: int = 100
    label_smoothing: float = 0.1

    # --- Generación ---
    num_beams: int = 4
    seed: int = 42

    def __post_init__(self):
        if self.language_pairs is None:
            self.language_pairs = [("es", "sw")]  # español -> suajili


config = Config()
print(config)

---
## 2. Carga de datos paralelos

Tres formatos soportados en `DATA_DIR`:

1. **CSV** — columnas `source`/`target` (o `src`/`tgt`)
2. **JSONL** — claves `source`/`target`
3. **Ficheros paralelos** — `train.src` + `train.tgt` (una frase por línea)

**PostgreSQL (opcional):** `load_from_postgres()` lee una tabla con columnas
`source`/`target` usando credenciales del entorno (`PGHOST`, `PGPORT`,
`PGUSER`, `PGPASSWORD`, `PGDATABASE`, `PGTABLE`) — nunca credenciales
hardcodeadas.

In [ ]:
# === CARGA DE DATOS ===
def load_parallel_pairs(data_dir: Path) -> List[Tuple[str, str]]:
    """Carga pares (source, target) desde CSV / JSONL / ficheros .src/.tgt."""
    data_dir = Path(data_dir)
    pairs: List[Tuple[str, str]] = []

    for csv_path in sorted(data_dir.glob("*.csv")):
        df = pd.read_csv(csv_path)
        col_s = "source" if "source" in df.columns else "src"
        col_t = "target" if "target" in df.columns else "tgt"
        for _, row in df.iterrows():
            if isinstance(row[col_s], str) and isinstance(row[col_t], str):
                pairs.append((row[col_s].strip(), row[col_t].strip()))

    for jsonl_path in sorted(data_dir.glob("*.jsonl")):
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                s = obj.get("source", obj.get("src"))
                t = obj.get("target", obj.get("tgt"))
                if s and t:
                    pairs.append((str(s).strip(), str(t).strip()))

    for src_file in sorted(data_dir.glob("*.src")):
        tgt_file = src_file.with_suffix(".tgt")
        if not tgt_file.exists():
            continue
        with open(src_file, encoding="utf-8") as fs, open(tgt_file, encoding="utf-8") as ft:
            for s, t in zip(fs, ft):
                s, t = s.strip(), t.strip()
                if s and t:
                    pairs.append((s, t))

    return list(dict.fromkeys(pairs))  # deduplicar preservando orden


def load_from_postgres(table: str = None) -> List[Tuple[str, str]]:
    """Carga pares desde PostgreSQL usando variables de entorno (opcional)."""
    from sqlalchemy import create_engine, text

    table = table or os.environ.get("PGTABLE", "nmt_pairs")
    conn_str = (
        f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
        f"@{os.environ.get('PGHOST', 'localhost')}:{os.environ.get('PGPORT', '5432')}"
        f"/{os.environ['PGDATABASE']}"
    )
    engine = create_engine(conn_str)
    with engine.connect() as conn:
        rows = conn.execute(text(f"SELECT source, target FROM {table}"))
        return [(str(s), str(t)) for s, t in rows]


# --- Cargar (ficheros primero; descomenta la línea de Postgres si aplica) ---
pairs = load_parallel_pairs(DATA_DIR)
# pairs = load_from_postgres()
print(f"📊 Pares cargados: {len(pairs)}")
if pairs:
    print(f"   Ejemplo: {pairs[0]}")

---
## 3. Limpieza de datos

Filtros aplicados:
- Frases vacías o demasiado cortas/largas (`min_len`/`max_len`)
- Duplicados exactos
- Normalización: minúsculas, espacios colapsados, puntuación estandarizada

In [ ]:
# === LIMPIEZA ===
def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)              # colapsar espacios
    text = re.sub(r"([.,!?;:])\1+", r"\1", text)  # puntuación repetida
    return text


def clean_pairs(pairs, min_len=3, max_len=75):
    cleaned = []
    for s, t in pairs:
        s, t = clean_text(s), clean_text(t)
        if not s or not t:
            continue
        if not (min_len <= len(s.split()) <= max_len):
            continue
        if not (min_len <= len(t.split()) <= max_len):
            continue
        cleaned.append((s, t))
    # Deduplicar y mezclar
    cleaned = list(dict.fromkeys(cleaned))
    random.shuffle(cleaned)
    return cleaned


pairs = clean_pairs(pairs, config.min_len, config.max_len)
print(f"✅ Tras limpieza: {len(pairs)} pares")

---
## 4. Direcciones y partición train/val/test

Si `bidirectional=True`, cada par genera dos muestras (`A→B` y `B→A`). Cada
muestra se etiqueta con su dirección (`es to sw`, `sw to es`, ...) que se
prefija al texto origen antes de tokenizar: `translate es to sw: <texto>`.

In [ ]:
# === DIRECCIONES + SPLIT ===
def expand_directions(pairs, language_pairs, bidirectional=True):
    """Devuelve (texto, dirección) para cada par y dirección configurada."""
    samples = []  # (source, target, direction)
    for src, tgt in language_pairs:
        for s, t in pairs:
            samples.append((s, t, f"{src} to {tgt}"))
            if bidirectional:
                samples.append((t, s, f"{tgt} to {src}"))
    return samples


samples = expand_directions(pairs, config.language_pairs, config.bidirectional)
random.Random(config.seed).shuffle(samples)

n_test = int(len(samples) * config.test_ratio)
n_val = int(len(samples) * config.val_ratio)
test_samples = samples[:n_test]
val_samples = samples[n_test:n_test + n_val]
train_samples = samples[n_test + n_val:]
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

---
## 5. Tokenización con ByT5

El tokenizador de ByT5 opera sobre **bytes UTF-8**, así que no necesita
vocabulario previo de la lengua. El texto origen se tokeniza con el prefijo
`translate {dirección}: `; el destino **sin** prefijo. En los labels, los
tokens de padding se sustituyen por `-100` para que no contribuyan a la loss.

In [ ]:
# === TOKENIZACIÓN ===
tokenizer = ByT5Tokenizer.from_pretrained(config.model_name)
print(f"Vocab ByT5: {tokenizer.vocab_size} | pad={tokenizer.pad_token_id} eos={tokenizer.eos_token_id}")


def tokenize_samples(samples, tokenizer, config):
    input_ids, attn_masks, labels, directions = [], [], [], []
    for s, t, direction in samples:
        prefix = f"translate {direction}: "
        enc = tokenizer(prefix + s, max_length=config.max_length,
                        truncation=True, return_tensors="pt")
        dec = tokenizer(t, max_length=config.max_length,
                        truncation=True, return_tensors="pt")
        lab = dec.input_ids.clone()
        lab[lab == tokenizer.pad_token_id] = -100
        input_ids.append(enc.input_ids.squeeze(0))
        attn_masks.append(enc.attention_mask.squeeze(0))
        labels.append(lab.squeeze(0))
        directions.append(direction)
    return input_ids, attn_masks, labels, directions


train_ids, train_mask, train_lab, train_dir = tokenize_samples(
    train_samples, tokenizer, config)
val_ids, val_mask, val_lab, val_dir = tokenize_samples(
    val_samples, tokenizer, config)
test_ids, test_mask, test_lab, test_dir = tokenize_samples(
    test_samples, tokenizer, config)
print(f"✅ Tokenizados: train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}")

---
## 6. Dataset y DataLoader

`NMTDataset` guarda los tensores ya tokenizados; `collate_fn` aplica padding
dentro de cada batch (mucho más eficiente que padding global).

In [ ]:
# === DATASET ===
class NMTDataset(Dataset):
    def __init__(self, input_ids, attn_masks, labels, directions):
        self.input_ids, self.attn_masks = input_ids, attn_masks
        self.labels, self.directions = labels, directions

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attn_masks[idx],
            "labels": self.labels[idx],
            "direction": self.directions[idx],
        }


def collate_fn(batch):
    pad = tokenizer.pad_token_id
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [b["input_ids"] for b in batch], batch_first=True, padding_value=pad)
    attn = torch.nn.utils.rnn.pad_sequence(
        [b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch], batch_first=True, padding_value=-100)
    return {"input_ids": input_ids, "attention_mask": attn,
            "labels": labels, "direction": [b["direction"] for b in batch]}


def make_loader(ids, mask, lab, dirs, shuffle):
    ds = NMTDataset(ids, mask, lab, dirs)
    return DataLoader(ds, batch_size=config.batch_size, shuffle=shuffle,
                      collate_fn=collate_fn, drop_last=shuffle)


train_loader = make_loader(train_ids, train_mask, train_lab, train_dir, shuffle=True)
val_loader = make_loader(val_ids, val_mask, val_lab, val_dir, shuffle=False)
print(f"Batches train: {len(train_loader)} | val: {len(val_loader)}")

---
## 7. Entrenamiento (PyTorch)

Bucle manual de entrenamiento con:
- `AdamW` + scheduler lineal con warmup
- `label_smoothing` en la loss (CrossEntropyLoss nativa de T5)
- Acumulación de gradientes
- Evaluación en validación cada epoch y guardado del **mejor** checkpoint

In [ ]:
# === MODELO + OPTIMIZADOR ===
model = T5ForConditionalGeneration.from_pretrained(config.model_name)
model.to(device)
print(f"Parámetros: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                              weight_decay=config.weight_decay)
total_steps = (len(train_loader) * config.num_epochs
               // config.gradient_accumulation_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, config.warmup_steps, total_steps)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, count = 0.0, 0
    for batch in loader:
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        total += out.loss.item()
        count += 1
    return total / max(count, 1)


def train_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total, steps = 0.0, 0
    optimizer.zero_grad()
    for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{config.num_epochs}"):
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        loss = out.loss / config.gradient_accumulation_steps
        loss.backward()
        total += out.loss.item()
        steps += 1

        if steps % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    return total / max(steps, 1)


# === BUCLE DE ENTRENAMIENTO ===
best_val = float("inf")
for epoch in range(config.num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch)
    val_loss = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(OUTPUT_DIR / "best")
        tokenizer.save_pretrained(OUTPUT_DIR / "best")
        print(f"💾 Mejor checkpoint: {OUTPUT_DIR / 'best'}")

model.save_pretrained(OUTPUT_DIR / "final")
tokenizer.save_pretrained(OUTPUT_DIR / "final")
print(f"✅ Modelo final en {OUTPUT_DIR / 'final'}")

---
## 8. Traducción

`translate()` recibe texto y una dirección (`"es to sw"`), aplica el prefijo y
decodifica con **beam search**.

In [ ]:
# === INFERENCIA ===
@torch.no_grad()
def translate(model, tokenizer, texts, direction, device, config):
    """Traduce una lista de textos en una dirección concreta."""
    if isinstance(texts, str):
        texts = [texts]
    prefix = f"translate {direction}: "
    enc = tokenizer([prefix + t for t in texts], padding=True, truncation=True,
                    max_length=config.max_length, return_tensors="pt").to(device)
    out = model.generate(
        **enc,
        max_new_tokens=config.max_length,
        num_beams=config.num_beams,
    )
    return tokenizer.batch_decode(out, skip_special_tokens=True)


# --- Prueba rápida ---
test_texts = ["Buenos días, ¿cómo estás?", "Me gusta aprender idiomas"]
for direccion in ["es to sw", "es to it"]:
    print(f"--- {direccion} ---")
    for t, trad in zip(test_texts, translate(model, tokenizer, test_texts, direccion, device, config)):
        print(f"  {t}  =>  {trad}")

---
## 9. Evaluación: BLEU y chrF

Se traduce el conjunto de **test** y se comparan hipótesis contra referencias
con **sacrebleu** (BLEU y chrF). `n_eval` limita el tamaño para pruebas rápidas.

In [ ]:
# === EVALUACIÓN (BLEU + chrF) ===
import sacrebleu


def evaluate_metrics(model, tokenizer, samples, device, config, n_eval=None):
    """BLEU/chrF por dirección sobre un subset de test."""
    if n_eval:
        samples = samples[:n_eval]
    refs_by_dir, hyps_by_dir = {}, {}
    for s, t, direction in samples:
        refs_by_dir.setdefault(direction, []).append(t)

    # Agrupar por dirección para traducir en lote
    for direction in refs_by_dir:
        srcs = [s for s, _, d in samples if d == direction]
        hyps = translate(model, tokenizer, srcs, direction, device, config)
        hyps_by_dir[direction] = hyps

    results = {}
    for direction, refs in refs_by_dir.items():
        hyps = hyps_by_dir[direction]
        bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
        chrf = sacrebleu.corpus_chrf(hyps, [refs]).score
        results[direction] = {"bleu": bleu, "chrf": chrf, "n": len(refs)}
    return results


results = evaluate_metrics(model, tokenizer, test_samples, device, config, n_eval=300)
print("=" * 46)
print(f"{'Dirección':<14}{'BLEU':>8}{'chrF':>8}{'N':>8}")
print("=" * 46)
for direction, r in results.items():
    print(f"{direction:<14}{r['bleu']:>8.2f}{r['chrf']:>8.2f}{r['n']:>8}")
print("=" * 46)

---
## 10. Publicar en Hugging Face Hub (opcional)

Requiere `HF_TOKEN` en el entorno (ver sección 0). Nunca pegues el token en el
notebook.

In [ ]:
# === PUSH AL HUB (opcional) ===
def push_to_hub(repo_id: str):
    token = os.environ.get("HF_TOKEN")
    if not token:
        print("⚠️ HF_TOKEN no definido. Ejecuta: export HF_TOKEN=hf_...")
        return
    model.push_to_hub(repo_id, token=token)
    tokenizer.push_to_hub(repo_id, token=token)
    print(f"🚀 Publicado en https://huggingface.co/{repo_id}")


# push_to_hub("tu-usuario/nmt-byt5-multilingue")  # <- descomenta para publicar
print("✅ Notebook completado")